In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)

# Define the file path for your CSV
FILE_PATH = '/content/drive/MyDrive/MScDS&AI/SVM/student-mat.csv'

# 1. Read the CSV file
# The provided snippet shows the delimiter is a semicolon
df = pd.read_csv(FILE_PATH, sep=';')
print("--- Successfully loaded data ---")


--- Successfully loaded data ---


In [13]:

# 2. Display columns and type
print("\n--- Columns and Data Types (Initial) ---")
# .info() gives a concise summary of columns, non-null counts, and dtypes
print(df.info())

# 3. Introduce a new column High_G3
# Condition: 'High' if G3 > 10, else 'Moderate'
# Using np.where as a simpler alternative to .apply()
df['High_G3'] = np.where(df['G3'] > 10, 'High', 'Moderate')
print("\n--- Created 'High_G3' column ---")

# 4. Print the distribution of number of samples in High_G3
print("\n--- Distribution of Target Variable (High_G3) ---")
print(df['High_G3'].value_counts())
print("Note: If the 'High' class has very few samples, the dataset will be")
print("highly imbalanced, which can make model training difficult.")



--- Columns and Data Types (Initial) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery 

In [14]:

# 5. Remove G3 column
df = df.drop('G3', axis=1)
print("\n--- Removed 'G3' column ---")

# 6. Prepare X (all features except High_G3)
X = df.drop('High_G3', axis=1)

# 7. Prepare y (High_G3)
y = df['High_G3']

# 8. Convert categorical features to one hot encoding using dummies
X_encoded = pd.get_dummies(X, drop_first=True)
print(f"\n--- Converted categorical features to dummies ---")
print(f"Original features: {len(X.columns)}, Features after dummies: {len(X_encoded.columns)}")

# 9. StandardScaler features
# Save column names *before* scaling
feature_names = X_encoded.columns

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
print("--- Applied StandardScaler to all features ---")




--- Removed 'G3' column ---

--- Converted categorical features to dummies ---
Original features: 32, Features after dummies: 41
--- Applied StandardScaler to all features ---


In [15]:

# 10. Perform train-test split of 80-20
# We use stratify=y to ensure proportional class representation
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n--- Data Split ---")
print(f"Training set size: {len(X_train)} (80%)")
print(f"Test set size: {len(X_test)} (20%)")
print("\nTraining set distribution:\n", y_train.value_counts(normalize=True))
print("\nTest set distribution:\n", y_test.value_counts(normalize=True))

# 11. Fit SVM classifier using kernel='rbf'
print("\n--- Training Model 1: Standard SVM (rbf) ---")
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train, y_train)

# Evaluate Model 1
y_pred_svm = svm_model.predict(X_test)

# --- Evaluation Metrics for: Standard SVM (rbf) ---
print(f"\n--- Evaluation Metrics for: Standard SVM (rbf) ---")
labels = ['High', 'Moderate']

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, labels=labels, target_names=labels))

print("Confusion Matrix:")
cm_1 = confusion_matrix(y_test, y_pred_svm, labels=labels)
print(f"               Predicted 'High' | Predicted 'Moderate'")
print(f"Actual 'High'      {cm_1[0, 0]:>10} | {cm_1[0, 1]:>18}")
print(f"Actual 'Moderate'  {cm_1[1, 0]:>10} | {cm_1[1, 1]:>18}")


# 12. Fit SVM classifier using kernel='rbf', class_weight='balanced'
print("\n--- Training Model 2: Balanced SVM (rbf, class_weight='balanced') ---")
svm_balanced_model = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_balanced_model.fit(X_train, y_train)

# Evaluate Model 2
y_pred_svm_balanced = svm_balanced_model.predict(X_test)

# --- Evaluation Metrics for: Balanced SVM (rbf, class_weight='balanced') ---
print(f"\n--- Evaluation Metrics for: Balanced SVM (rbf, class_weight='balanced') ---")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm_balanced, labels=labels, target_names=labels))

print("Confusion Matrix:")
cm_2 = confusion_matrix(y_test, y_pred_svm_balanced, labels=labels)
print(f"               Predicted 'High' | Predicted 'Moderate'")
print(f"Actual 'High'      {cm_2[0, 0]:>10} | {cm_2[0, 1]:>18}")
print(f"Actual 'Moderate'  {cm_2[1, 0]:>10} | {cm_2[1, 1]:>18}")


--- Data Split ---
Training set size: 316 (80%)
Test set size: 79 (20%)

Training set distribution:
 High_G3
High        0.528481
Moderate    0.471519
Name: proportion, dtype: float64

Test set distribution:
 High_G3
High        0.531646
Moderate    0.468354
Name: proportion, dtype: float64

--- Training Model 1: Standard SVM (rbf) ---

--- Evaluation Metrics for: Standard SVM (rbf) ---

Classification Report:
              precision    recall  f1-score   support

        High       0.88      0.88      0.88        42
    Moderate       0.86      0.86      0.86        37

    accuracy                           0.87        79
   macro avg       0.87      0.87      0.87        79
weighted avg       0.87      0.87      0.87        79

Confusion Matrix:
               Predicted 'High' | Predicted 'Moderate'
Actual 'High'              37 |                  5
Actual 'Moderate'           5 |                 32

--- Training Model 2: Balanced SVM (rbf, class_weight='balanced') ---

--- Evaluat